# LLM 7일차 (2) — 조건부 분기 에이전트 : 질문 분류 → (필요할 때만) Tavily 웹 검색 → 답변

수업 필기 `LangGraph_실습3` 을 정리 · 완성한 노트입니다. 필기는 노드 3개까지 쓰고 **'라우팅 함수 정의' 에서 멈춰** 있어서, 교재 p.100~101 의 흐름대로 라우팅 함수 · 그래프 구성 · 실행을 채웠습니다.
(교재 `AI Agent` **p.100~102** — 실습 개요 7단계 · 작업 흐름)

```
[사용자 질문] → classifier_node ── needs_search=True  (최신 정보 · 실시간 실적) ──→ search_node ──→ answer_node → END
                    (SEARCH / NO_SEARCH 판별)  └─ needs_search=False (일반 상식 · 인사말) ──────────────┘
```

- 조건부 엣지의 기본 동작(매핑 · 오타 · 반복)은 [`langgraph_conditional_edges.ipynb`](./langgraph_conditional_edges.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 필기 원본에는 실행 출력이 없다(라우팅 함수 정의에서 멈춤). 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# ⏳` | OpenAI · Tavily 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | 진짜 LLM · 검색 대신 **가짜 LLM · 가짜 검색 도구**로 실제로 돌린 셀 — 분류 · 답변 문장은 gpt-5-nano 와 다르다. **그래프가 고르는 길 · 상태 변화 · 오류는 진짜와 같다** |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langgraph` | `StateGraph` · `END` · `add_messages` | `pip install langgraph` |
| `langchain-openai` | `ChatOpenAI(model='gpt-5-nano')` | `pip install langchain-openai` |
| `langchain-tavily` | `TavilySearch(max_results=3)` — 웹 검색 도구 | `pip install langchain-tavily` |
| `python-dotenv` | `.env` 에서 키 읽기 | `pip install python-dotenv` |

1. 노트북과 같은 폴더(또는 위 폴더)의 `.env` 에 `OPENAI_API_KEY=…` · `TAVILY_API_KEY=…` 를 적는다 (**`.env` 는 올리지 않는다** — 저장소 `.gitignore` 에 포함)
2. 맨 위부터 실행. **키가 없으면** `# ⏳` 셀을 건너뛰고 `# 🧪` 셀(가짜 LLM · 가짜 검색)로 끝까지 실행된다
3. 진짜로 돌리려면 `# ⏳ 진짜 LLM · 검색 도구` 셀을 실행하고 `# 🧪` 셀은 건너뛴 뒤, 아래로 계속 (정리 환경 : `langgraph 1.2.21` · `langchain-openai 1.7.4` · `langchain-tavily 0.2.18`)

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph langchain-openai langchain-tavily python-dotenv

## 1. 환경 설정 (교재 p.100 의 1 · 3단계)

In [2]:
import os
from typing import Annotated, Literal
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import ToolException
from langgraph.graph import StateGraph, END, add_messages

In [3]:
# 1) API 키 로드 — .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
loaded = load_dotenv(find_dotenv(usecwd=True))
print('.env 읽음 :', loaded,
      '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')),
      '| TAVILY_API_KEY 있음 :', bool(os.getenv('TAVILY_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False | TAVILY_API_KEY 있음 : False


> **📝 고친 부분** 필기는 `from dotenv import load_dotenv` 만 하고 **`load_dotenv()` 를 부르지 않았다**. 그러면 `.env` 의 키가 환경 변수로 들어가지 않아 `ChatOpenAI(...)` · `TavilySearch(...)` 를 만드는 순간 '키를 찾을 수 없다' 는 오류가 난다(아래 확인). 교재 p.100 실습 개요의 1단계 'API 키 로드' 를 채웠다.

In [4]:
# 키 없이 TavilySearch 를 만들면? (정리 환경에는 키가 없다)
from langchain_tavily import TavilySearch
try:
    TavilySearch(max_results=3)
except Exception as e:
    print(type(e).__name__, '—', str(e).splitlines()[1].strip()[:110])

ValidationError — Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains i


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 3) 진짜 LLM · 검색 도구 (키가 있을 때)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5-nano')
search_tool = TavilySearch(max_results=3)

In [6]:
# 🧪 OpenAI · Tavily 대신 '정해 둔 규칙으로 답하는' 가짜 LLM · 가짜 검색 도구 — 그래프가 어느 길로 가는지 확인용

# (진짜 키로 실행할 때는 이 셀을 건너뛴다)
from langchain_core.runnables import RunnableLambda

SEARCH_WORDS = ('2026', '최신', '오늘', '실적', '뉴스', '현재')

def _fake_llm(messages):
    system, user = messages[0][1], messages[-1][1]
    if 'NO_SEARCH' in system:                                   # classifier_node 의 질문
        return AIMessage('SEARCH' if any(w in user for w in SEARCH_WORDS) else 'NO_SEARCH')
    if '[웹 검색 결과]' in system:                               # answer_node — 검색 결과를 받은 경우
        n = system.count('출처:')
        return AIMessage(f'(모의 답변 · 웹 검색 결과 {n}개 반영) {user}')
    return AIMessage(f'(모의 답변 · 모델 자체 지식) {user}')

def _fake_search(args):
    q = args['query']
    if '찾을 수 없는' in q:                                      # TavilySearch 는 결과가 0개면 ToolException 을 던진다
        raise ToolException(f"No search results found for '{q}'.")
    return {'query': q, 'results': [
        {'url': f'https://example.com/news/{i}', 'title': f'모의 기사 {i}', 'content': f"'{q}' 에 대한 모의 검색 결과 {i}", 'score': 0.9 - i / 10}
        for i in range(1, 4)]}

llm = RunnableLambda(_fake_llm)
search_tool = RunnableLambda(_fake_search)
print(search_tool.invoke({'query': '테스트'})['results'][0])

{'url': 'https://example.com/news/1', 'title': '모의 기사 1', 'content': "'테스트' 에 대한 모의 검색 결과 1", 'score': 0.8}


- 진짜 `TavilySearch.invoke({'query': …})` 도 같은 모양의 **dict** 를 돌려준다 — `{'query', 'results': [{'url', 'title', 'content', 'score', …}], 'response_time', …}`

## 2. 상태 정의 — Pydantic `BaseModel` (교재 p.100 의 2단계)

In [7]:
# Pydantic 라이브러리의 BaseModel 클래스 상속 -> 커스텀 자료형(상태) 생성
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []   # 대화 기록 — 덧붙이기
    needs_search: bool = False                                  # 웹 검색이 필요한가 (True / False)
    search_results: str = ''                                    # Tavily 검색 결과를 합친 글

print(AgentState(messages=[HumanMessage('안녕')]))

messages=[HumanMessage(content='안녕', additional_kwargs={}, response_metadata={})] needs_search=False search_results=''


> **📝 고친 부분** 주석 오타 `Pydanctic` → `Pydantic`. `= []` 기본값은 Pydantic 이 객체마다 **복사**하므로 일반 클래스와 달리 공유 문제가 없다(`Field(default_factory=list)` 와 같은 효과).

## 3. 노드 함수 (교재 p.100 의 4단계)

### 3-1. `classifier_node` — 검색이 필요한가? (SEARCH / NO_SEARCH)

In [8]:
CLASSIFIER_PROMPT = '''
당신은 사용자의 질문을 분석하는 전문가입니다.
사용자의 질문에 답변하기 위해 실시간 웹 검색이 반드시 필요한지 판단하세요.

- 최신 시사 뉴스, 실시간 정보, 특정 연도 실적(예: 2026년 실적) 등 최신 데이터 조회가 필요한 경우: 'SEARCH'
- 일반 상식, 개념 설명, 프로그래밍 문법, 단순 인사말 등 모델 자체 지식으로 충분한 경우: 'NO_SEARCH'

반드시 'SEARCH' 또는 'NO_SEARCH' 단어 하나만 정확하게 출력하세요. 다른 설명은 일체 포함하지 마세요.
'''

def parse_decision(text: str) -> bool:
    # 'SEARCH' · "SEARCH." · ' search ' 처럼 따옴표 · 마침표 · 공백 · 대소문자가 섞여도 같은 뜻으로
    return text.strip().strip('\'"`.').strip().upper() == 'SEARCH'

def classifier_node(state: AgentState) -> dict:
    print('[Node 1: 질문 의도 분류]')
    user_question = state.messages[-1].content                 # 대화 기록의 마지막 = 사용자 질문
    response = llm.invoke([('system', CLASSIFIER_PROMPT), ('user', f'질문: {user_question}')])
    needs_search = parse_decision(response.content)            # 단답 문자열 → 불리언
    print(f'  질문 : {user_question}\n  LLM 판단 : {response.content!r} → needs_search = {needs_search}')
    return {'needs_search': needs_search}                      # 바꿀 항목만 돌려준다

> **📝 고친 부분** 필기는 `decision == 'SEARCH'` 로 비교했다. 그런데 프롬프트가 **`'SEARCH'` 처럼 따옴표를 붙여** 보여 주기 때문에 LLM 이 따옴표까지 따라 쓰면 `False` → **검색이 필요한 질문인데 검색을 건너뛴다**. 반대로 `'SEARCH' in decision` 으로 느슨하게 바꾸면 **`NO_SEARCH` 안에도 `SEARCH` 가 들어 있어** 전부 검색으로 간다. 아래에서 세 방법을 비교했다.

In [9]:
samples = ['SEARCH', "'SEARCH'", 'Search.', 'NO_SEARCH', "'NO_SEARCH'", ' search\n']
print(f"{'LLM 출력':<16}{'== (필기)':<12}{'in (느슨)':<12}{'parse_decision':<14}")
for s in samples:
    d = s.strip().upper()
    print(f'{s!r:<16}{str(d == "SEARCH"):<12}{str("SEARCH" in d):<12}{str(parse_decision(s)):<14}')

LLM 출력          == (필기)     in (느슨)     parse_decision
'SEARCH'        True        True        True          
"'SEARCH'"      False       True        True          
'Search.'       False       True        True          
'NO_SEARCH'     False       True        False         
"'NO_SEARCH'"   False       True        False         
' search\n'     True        True        True          


- `==` 는 따옴표 · 마침표가 붙으면 놓치고(False), `in` 은 `NO_SEARCH` 를 잘못 잡는다(True). 앞뒤 따옴표 · 마침표를 떼고 `==` 로 비교하는 것이 안전하다
- 더 확실하게 하려면 `llm.with_structured_output(…)` 으로 `Literal['SEARCH', 'NO_SEARCH']` 같은 **정해진 형식**으로 받는다 (Day 051 Pydantic 의 쓰임)

### 3-2. `tavily_search_node` — 웹 검색 결과를 글 하나로

In [10]:
# 필기 원본 그대로 (비교용) — 오타 주석만 정리
def tavily_search_node_original(state: AgentState) -> dict:
    query = state.messages[-1].content
    try:
        raw_results = search_tool.invoke({'query': query})
        results = raw_results.get('results', []) if isinstance(raw_results, dict) else []
        context_parts = [f"출처: {r.get('url', '')}\n내용: {r.get('content', '')}" for r in results]
        if context_parts:
            combined_results = '\n\n'.join(context_parts)
        else:
            combined_results = '검색 결과가 없습니다.'
    except Exception as e:
        error_message = f'검색 중 오류 발생: {e}'
        print(error_message)
    return {'search_results': combined_results}          # ← except 로 오면 combined_results 가 없다

try:
    tavily_search_node_original(AgentState(messages=[HumanMessage('찾을 수 없는 아주 이상한 질문')]))
except Exception as e:
    print('→', type(e).__name__, ':', e)

검색 중 오류 발생: No search results found for '찾을 수 없는 아주 이상한 질문'.
→ UnboundLocalError : cannot access local variable 'combined_results' where it is not associated with a value


> **📝 고친 부분** 필기는 `except` 에서 `error_message` 만 만들고 `combined_results` 를 정하지 않아, 검색이 실패하면 오류를 잡은 **다음 줄에서 `UnboundLocalError`** 가 난다(위 출력). 진짜 `TavilySearch` 는 **결과가 0개일 때 `ToolException` 을 던지므로**(langchain-tavily 소스 확인) 드문 일이 아니다. 또 네트워크 오류 같은 다른 예외는 던지지 않고 **`{'error': …}` dict 를 돌려주는데**, 필기 코드는 이것을 '검색 결과가 없습니다' 로 처리해 오류가 가려진다. 두 경우를 모두 고쳤다.

In [11]:
def tavily_search_node(state: AgentState) -> dict:
    print('[Node 2: Tavily 웹 검색]')
    query = state.messages[-1].content                          # 대화 기록(히스토리)의 마지막 질문
    combined_results = '검색 결과가 없습니다.'                    # 📝 먼저 기본값 — 어느 길로 가도 정의됨
    try:
        raw_results = search_tool.invoke({'query': query})       # dict → 'results' 키 → 검색 결과 dict 리스트
        if isinstance(raw_results, dict) and 'error' in raw_results:
            combined_results = f"검색 중 오류 발생: {raw_results['error']}"   # 📝 TavilySearch 가 오류를 dict 로 돌려준 경우
        else:
            results = raw_results.get('results', []) if isinstance(raw_results, dict) else []
            context_parts = [f"출처: {r.get('url', '')}\n내용: {r.get('content', '')}"
                             if isinstance(r, dict) else str(r) for r in results]
            if context_parts:
                combined_results = '\n\n'.join(context_parts)      # 출처 + 본문 3개 → 글 하나
    except Exception as e:
        combined_results = f'검색 중 오류 발생: {e}'             # 📝 오류 내용을 상태에 남긴다
    print(f'  검색어 : {query}\n  결과 : {combined_results.splitlines()[0][:70]} … ({len(combined_results)}자)')
    return {'search_results': combined_results}

for q in ['2026년 엔비디아 2분기 실적', '찾을 수 없는 아주 이상한 질문']:
    tavily_search_node(AgentState(messages=[HumanMessage(q)])); print()

[Node 2: Tavily 웹 검색]
  검색어 : 2026년 엔비디아 2분기 실적
  결과 : 출처: https://example.com/news/1 … (214자)

[Node 2: Tavily 웹 검색]
  검색어 : 찾을 수 없는 아주 이상한 질문
  결과 : 검색 중 오류 발생: No search results found for '찾을 수 없는 아주 이상한 질문'. … (60자)



### 3-3. `answer_node` — 검색 결과가 있으면 프롬프트에 넣어서

In [12]:
def answer_node(state: AgentState) -> dict:
    print('[Node 3: 최종 답변 생성]')
    user_question = state.messages[-1].content
    if state.needs_search and state.search_results:            # 검색을 거쳐 온 경우
        system_prompt = f'''당신은 정확한 최신 정보를 제공하는 전문 AI 비서입니다.
아래 제공된 [웹 검색 결과]를 바탕으로 사용자의 질문에 친절하고 명확하게 답변해 주세요.

[웹 검색 결과]
{state.search_results}'''
    else:                                                      # 일반 질문
        system_prompt = '당신은 친절하고 똑똑한 AI 비서입니다. 사용자의 질문에 명쾌하게 답변해 주세요.'
    response = llm.invoke([('system', system_prompt), ('user', user_question)])
    print(f'  답변 : {response.content[:80]}')
    return {'messages': [AIMessage(content=response.content)]}  # add_messages → 대화 기록에 덧붙음

- 필기 주석 오타 `히스토뢰` → 히스토리 · `블리언` → 불리언 · `'reuslts'` → `'results'` 도 정리했다

## 4. 라우팅 함수 (교재 p.100 의 5단계 · p.101 `route_by_search_need`)

필기가 멈춘 곳이다. **상태의 `needs_search` 를 읽고 다음 노드의 이름을 돌려준다.**

In [13]:
def route_by_search_need(state: AgentState) -> Literal['search_node', 'answer_node']:
    if state.needs_search:
        return 'search_node'        # 최신 정보 · 실시간 실적 → 웹 검색
    return 'answer_node'            # 일반 상식 · 인사말 → 바로 답변

print(route_by_search_need(AgentState(needs_search=True)), '/', route_by_search_need(AgentState(needs_search=False)))

search_node / answer_node


## 5. 그래프 구성 · 컴파일 (교재 p.100 의 6단계 · p.101)

In [14]:
workflow = StateGraph(AgentState)                                # 그래프 생성

workflow.add_node('classifier_node', classifier_node)           # 노드 추가 — '이름', 함수
workflow.add_node('search_node', tavily_search_node)            # 이름(search_node) ≠ 함수 이름(tavily_search_node)
workflow.add_node('answer_node', answer_node)

workflow.set_entry_point('classifier_node')                     # 시작점 = add_edge(START, 'classifier_node')

workflow.add_conditional_edges(                                 # 조건부 엣지
    'classifier_node',                                          #   1단계 : 어디서 출발
    route_by_search_need,                                       #   2단계 : 무엇으로 판단
    {'search_node': 'search_node', 'answer_node': 'answer_node'},   # 3단계 : 어디로 보낼지
)
workflow.add_edge('search_node', 'answer_node')                 # 검색 뒤에는 항상 답변
workflow.add_edge('answer_node', END)                           # 📝 교재 p.101 의 work.add_edge → workflow.add_edge

agent = workflow.compile()
print(type(agent).__name__)
print(agent.get_graph().draw_mermaid())

CompiledStateGraph
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classifier_node(classifier_node)
	search_node(search_node)
	answer_node(answer_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> classifier_node;
	classifier_node -.-> answer_node;
	classifier_node -.-> search_node;
	search_node --> answer_node;
	answer_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



> **📝 교재 p.101 보충**
> - 그림 아래 코드 `work.add_edge("answer_node", END)` 는 변수 이름 오타 → `workflow.add_edge(...)` (그대로 쓰면 `NameError: name 'work' is not defined`)
> - 그림의 상자에는 `tavily_search_node` 라고 적혀 있지만 엣지 코드는 `"search_node"` 를 쓴다. 엣지 · 매핑에 쓰는 것은 **`add_node` 에 준 노드 이름**이므로 `workflow.add_node('search_node', tavily_search_node)` 로 등록해야 교재 코드가 그대로 동작한다

## 6. 실행 — 질문마다 다른 길 (교재 p.100 의 7단계)

`stream(stream_mode='updates')` 는 노드가 끝날 때마다 `{노드 이름: 바꾼 항목}` 을 준다 — **어느 길로 갔는지** 바로 보인다.

In [15]:
# 🧪 OpenAI · Tavily 대신 '정해 둔 규칙으로 답하는' 가짜 LLM · 가짜 검색 도구 — 그래프가 어느 길로 가는지 확인용

questions = ['2026년 엔비디아 2분기 실적 알려줘', '파이썬 리스트 컴프리헨션이 뭐야?', '안녕하세요!']
for q in questions:
    print('=' * 80, f'\n질문 : {q}')
    path = []
    for update in agent.stream({'messages': [HumanMessage(content=q)]}, stream_mode='updates'):
        node, changed = next(iter(update.items()))
        path.append(node)
    print('▶ 실행 경로 :', ' → '.join(path))

질문 : 2026년 엔비디아 2분기 실적 알려줘
[Node 1: 질문 의도 분류]
  질문 : 2026년 엔비디아 2분기 실적 알려줘
  LLM 판단 : 'SEARCH' → needs_search = True
[Node 2: Tavily 웹 검색]
  검색어 : 2026년 엔비디아 2분기 실적 알려줘
  결과 : 출처: https://example.com/news/1 … (226자)
[Node 3: 최종 답변 생성]
  답변 : (모의 답변 · 웹 검색 결과 3개 반영) 2026년 엔비디아 2분기 실적 알려줘
▶ 실행 경로 : classifier_node → search_node → answer_node
질문 : 파이썬 리스트 컴프리헨션이 뭐야?
[Node 1: 질문 의도 분류]
  질문 : 파이썬 리스트 컴프리헨션이 뭐야?
  LLM 판단 : 'NO_SEARCH' → needs_search = False
[Node 3: 최종 답변 생성]
  답변 : (모의 답변 · 모델 자체 지식) 파이썬 리스트 컴프리헨션이 뭐야?
▶ 실행 경로 : classifier_node → answer_node
질문 : 안녕하세요!
[Node 1: 질문 의도 분류]
  질문 : 안녕하세요!
  LLM 판단 : 'NO_SEARCH' → needs_search = False
[Node 3: 최종 답변 생성]
  답변 : (모의 답변 · 모델 자체 지식) 안녕하세요!
▶ 실행 경로 : classifier_node → answer_node


- 검색이 필요한 질문은 `classifier_node → search_node → answer_node`, 일반 질문은 `classifier_node → answer_node` — **같은 그래프가 질문에 따라 다른 길**로 간다

### 최종 상태 — `invoke()` 결과

In [16]:
# 🧪 OpenAI · Tavily 대신 '정해 둔 규칙으로 답하는' 가짜 LLM · 가짜 검색 도구 — 그래프가 어느 길로 가는지 확인용

for q in ['2026년 엔비디아 2분기 실적 알려줘', '안녕하세요!']:
    result = agent.invoke({'messages': [HumanMessage(content=q)]})
    print('-' * 80)
    print('결과 자료형 :', type(result).__name__, '| 키 :', list(result))
    print('needs_search :', result['needs_search'], '| search_results :', repr(result.get('search_results', '(키 없음)'))[:50])
    print('마지막 메시지 :', result['messages'][-1].content)

[Node 1: 질문 의도 분류]
  질문 : 2026년 엔비디아 2분기 실적 알려줘
  LLM 판단 : 'SEARCH' → needs_search = True
[Node 2: Tavily 웹 검색]
  검색어 : 2026년 엔비디아 2분기 실적 알려줘
  결과 : 출처: https://example.com/news/1 … (226자)
[Node 3: 최종 답변 생성]
  답변 : (모의 답변 · 웹 검색 결과 3개 반영) 2026년 엔비디아 2분기 실적 알려줘
--------------------------------------------------------------------------------
결과 자료형 : dict | 키 : ['messages', 'needs_search', 'search_results']
needs_search : True | search_results : "출처: https://example.com/news/1\n내용: '2026년 엔비디아 2
마지막 메시지 : (모의 답변 · 웹 검색 결과 3개 반영) 2026년 엔비디아 2분기 실적 알려줘
[Node 1: 질문 의도 분류]
  질문 : 안녕하세요!
  LLM 판단 : 'NO_SEARCH' → needs_search = False
[Node 3: 최종 답변 생성]
  답변 : (모의 답변 · 모델 자체 지식) 안녕하세요!
--------------------------------------------------------------------------------
결과 자료형 : dict | 키 : ['messages', 'needs_search']
needs_search : False | search_results : '(키 없음)'
마지막 메시지 : (모의 답변 · 모델 자체 지식) 안녕하세요!


- `invoke()` 결과는 BaseModel 이 아니라 **dict** 다 (Day 051 확인)
- 📝 검색을 건너뛴 질문의 결과에는 **`search_results` 키가 아예 없다**. 기본값(`''`)은 노드가 받는 `AgentState` 객체 안에만 있고, 결과 dict 에는 **실행 중에 한 번이라도 쓰인 항목만** 들어온다 → `result['search_results']` 대신 `result.get('search_results', '')`

### 검색이 실패해도 그래프는 끝까지 간다

In [17]:
# 🧪 OpenAI · Tavily 대신 '정해 둔 규칙으로 답하는' 가짜 LLM · 가짜 검색 도구 — 그래프가 어느 길로 가는지 확인용

result = agent.invoke({'messages': [HumanMessage(content='2026년 찾을 수 없는 아주 이상한 실적')]})
print('search_results :', result['search_results'][:80])

[Node 1: 질문 의도 분류]
  질문 : 2026년 찾을 수 없는 아주 이상한 실적
  LLM 판단 : 'SEARCH' → needs_search = True
[Node 2: Tavily 웹 검색]
  검색어 : 2026년 찾을 수 없는 아주 이상한 실적
  결과 : 검색 중 오류 발생: No search results found for '2026년 찾을 수 없는 아주 이상한 실적'. … (66자)
[Node 3: 최종 답변 생성]
  답변 : (모의 답변 · 웹 검색 결과 0개 반영) 2026년 찾을 수 없는 아주 이상한 실적
search_results : 검색 중 오류 발생: No search results found for '2026년 찾을 수 없는 아주 이상한 실적'.


- 필기 원본 노드였다면 여기서 `UnboundLocalError` 로 그래프 전체가 멈췄다. 고친 노드는 오류 내용을 `search_results` 에 남기고 `answer_node` 로 넘어간다
- 📝 다만 `answer_node` 의 조건 `state.needs_search and state.search_results` 는 '검색 중 오류 발생 …' 이라는 글도 **검색 결과로 여겨** 프롬프트에 넣는다(위 출력의 '웹 검색 결과 0개 반영'). 오류일 때 일반 답변으로 돌리려면 상태에 `search_ok: bool` 같은 항목을 따로 두는 편이 낫다

## 7. ⏳ 진짜 gpt-5-nano + Tavily 로 실행

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

llm = ChatOpenAI(model='gpt-5-nano')
search_tool = TavilySearch(max_results=3)
# 노드 함수는 전역 llm · search_tool 을 쓰므로 그래프를 다시 만들 필요 없다
for q in ['2026년 엔비디아 2분기 실적 알려줘', '파이썬 리스트 컴프리헨션이 뭐야?', '안녕하세요!']:
    print('=' * 80, f'\n질문 : {q}')
    path = [next(iter(u)) for u in agent.stream({'messages': [HumanMessage(content=q)]}, stream_mode='updates')]
    print('▶ 실행 경로 :', ' → '.join(path))

- 직접 실행해 볼 것 : gpt-5-nano 가 '2026년' 이 없는 시사 질문(예 : '오늘 코스피 지수')도 SEARCH 로 고르는가 · 분류 LLM 출력에 따옴표가 붙는 일이 있는가(`LLM 판단 : …` 줄)

## 정리

| 단계 (교재 p.100) | 코드 |
|---|---|
| 1) API 키 로드 | `load_dotenv(find_dotenv(usecwd=True))` — **불러야** 적용된다 |
| 2) 상태 | `AgentState(BaseModel)` : `messages` (add_messages) · `needs_search: bool` · `search_results: str` |
| 3) LLM · 도구 | `ChatOpenAI(model='gpt-5-nano')` · `TavilySearch(max_results=3)` → dict `{'results': [...]}` |
| 4) 노드 | 분류(SEARCH/NO_SEARCH → bool) · 검색(결과 3개 → 글 하나, **실패해도 값 정의**) · 답변(검색 결과 유무로 프롬프트 바꿈) |
| 5) 라우팅 함수 | `route_by_search_need(state) -> Literal['search_node', 'answer_node']` |
| 6) 그래프 | `set_entry_point` → `add_conditional_edges(분류, 라우팅, 매핑)` → `add_edge(search, answer)` → `add_edge(answer, END)` → `compile()` |
| 7) 실행 | `stream(stream_mode='updates')` 로 경로 확인 · `invoke()` 결과는 dict (쓰인 항목만) |